# Policy-noise planning: keep search near a policy's action support

**Evidence label: illustrative toy experiment. This is not a LeWAM implementation or paper reproduction.**

A world model can be wrong. A planner that searches arbitrary actions can find actions that look useful to the model but do not work in the environment. Diffusion steering changes the planner's search variable: it searches the noise that a generative policy maps to actions.

In the paper, the policy maps Gaussian noise to an action chunk. The planner rolls out each decoded chunk in the imagined state, scores the final predicted goal distance, and executes the first chunk before replanning. The authors also project noise proposals onto the policy sampler's typical set. [Read the primary paper](https://arxiv.org/abs/2610.12407).

This notebook uses a hand-authored one-dimensional toy policy and an intentionally inaccurate forward model. It isolates the search-variable idea. It does not test whether the paper's method works on robotics tasks.


## The two search paths

```text
Raw-action search:  candidate actions ──> same toy forward model ──> predicted goal score
Policy-noise search: candidate noise ──> toy policy ──> same forward model ──> predicted goal score

Both methods execute the first action, observe the toy state, and plan again.
```

The policy emits actions on the relation `auxiliary = 0.5 × drive`. The toy environment only responds to `drive`. The forward model adds false progress when an action breaks the policy relation. This error is deliberate so the notebook can show one way a planner may exploit a model. It is not a claim about how often this happens in real systems.


In [ ]:
import math
import platform
import random
import statistics
import sys
import time

GOAL = 1.0
SUCCESS_THRESHOLD = 0.98
MAX_DRIVE = 0.5
MODEL_BIAS = 4.0
EFFORT_WEIGHT = 0.02
PLANNER_CANDIDATES = 128
PLANNING_HORIZON = 3
MAX_EPISODE_STEPS = 8
EVALUATION_SEEDS = (101, 202, 303, 404, 505)
EPISODES_PER_SEED = 20

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print("Device: CPU; this notebook uses the Python standard library only.")
print(f"Evaluation: {len(EVALUATION_SEEDS)} seeds × {EPISODES_PER_SEED} paired starts per seed")
print(f"Planner budget: {PLANNER_CANDIDATES} candidate plans × horizon {PLANNING_HORIZON} per decision")


## Toy policy, environment, and model

Let `x` be the position, and let an action be `(drive, auxiliary)`.

- The toy policy maps standard-normal noise `ε` to `drive = 0.5 × sigmoid(ε)` and then sets `auxiliary = 0.5 × drive`.
- The toy environment advances by `drive` and ignores `auxiliary`.
- The toy forward model predicts the same movement on policy actions. Off the policy relation, it adds a false progress term.

The rule controller below sees `x` directly. It is a simple reference for this toy task, not a matched baseline from the paper.


In [ ]:
def clip_position(value):
    return min(1.0, max(0.0, value))


def policy_from_noise(epsilon):
    drive = MAX_DRIVE / (1.0 + math.exp(-epsilon))
    return drive, 0.5 * drive


def environment_step(position, action):
    drive, _auxiliary = action
    return clip_position(position + drive)


def forward_model_step(position, action):
    drive, auxiliary = action
    off_support = max(auxiliary - 0.5 * drive, 0.0)
    predicted_progress = drive + MODEL_BIAS * off_support
    return clip_position(position + predicted_progress)


def sample_action(mode, rng):
    if mode == "raw_action":
        # Independent dimensions include actions the toy policy never emits.
        return rng.uniform(0.0, MAX_DRIVE), rng.uniform(0.0, MAX_DRIVE)
    if mode == "policy_noise":
        return policy_from_noise(rng.gauss(0.0, 1.0))
    raise ValueError(f"Unknown planner mode: {mode}")


def choose_first_action(position, mode, rng):
    best_score = math.inf
    best_first_action = None
    best_first_prediction = None

    for _candidate in range(PLANNER_CANDIDATES):
        predicted_position = position
        action_energy = 0.0
        first_action = None
        first_prediction = None

        for step_index in range(PLANNING_HORIZON):
            action = sample_action(mode, rng)
            predicted_position = forward_model_step(predicted_position, action)
            action_energy += action[0] ** 2 + action[1] ** 2
            if step_index == 0:
                first_action = action
                first_prediction = predicted_position

        score = (GOAL - predicted_position) ** 2 + EFFORT_WEIGHT * action_energy
        if score < best_score:
            best_score = score
            best_first_action = first_action
            best_first_prediction = first_prediction

    model_calls = PLANNER_CANDIDATES * PLANNING_HORIZON
    return best_first_action, best_first_prediction, model_calls


def direct_control_action(position):
    drive = min(MAX_DRIVE, GOAL - position)
    return drive, 0.5 * drive

print("Policy action at ε=0:", tuple(round(x, 3) for x in policy_from_noise(0.0)))
print("Model prediction for an on-support action:", round(forward_model_step(0.0, (0.4, 0.2)), 3))
print("Model prediction for an off-support action:", round(forward_model_step(0.0, (0.0, 0.25)), 3))
print("Environment result for that off-support action:", round(environment_step(0.0, (0.0, 0.25)), 3))


## Run the paired closed-loop evaluation

Each evaluation seed creates 20 shared start positions. The planner methods use the same candidate count, horizon, score, and eight-decision limit. The rule controller uses the same starts and episode limit, but it does not call the forward model.

The five seeds measure variation in sampled task starts and planner proposals. There are no model-training seeds: no model is trained in this example.


In [ ]:
def run_episode(method, start, planner_seed):
    position = start
    rng = random.Random(planner_seed)
    model_calls = 0
    prediction_errors = []
    off_support_actions = []
    steps = 0

    for _ in range(MAX_EPISODE_STEPS):
        if position >= SUCCESS_THRESHOLD:
            break

        if method == "direct_control":
            action = direct_control_action(position)
        else:
            mode = "raw_action" if method == "raw_action" else "policy_noise"
            action, predicted_next, calls = choose_first_action(position, mode, rng)
            model_calls += calls
            actual_next = environment_step(position, action)
            prediction_errors.append(abs(predicted_next - actual_next))
            off_support_actions.append(action[1] > 0.5 * action[0] + 1e-12)

        position = environment_step(position, action)
        steps += 1

    return {
        "method": method,
        "success": position >= SUCCESS_THRESHOLD,
        "steps": steps,
        "model_calls": model_calls,
        "final_position": position,
        "mean_selected_action_error": statistics.mean(prediction_errors) if prediction_errors else None,
        "off_support_fraction": statistics.mean(off_support_actions) if off_support_actions else None,
    }

methods = ("raw_action", "policy_noise", "direct_control")
results = []
run_started = time.perf_counter()

for evaluation_seed in EVALUATION_SEEDS:
    start_rng = random.Random(10_000 + evaluation_seed)
    starts = [start_rng.uniform(0.0, 0.2) for _ in range(EPISODES_PER_SEED)]
    for start_id, start in enumerate(starts):
        for method_index, method in enumerate(methods):
            planner_seed = 1_000_000 + evaluation_seed * 1000 + start_id * 10 + method_index
            record = run_episode(method, start, planner_seed)
            record.update({"evaluation_seed": evaluation_seed, "start_id": start_id, "start": start})
            results.append(record)

elapsed_seconds = time.perf_counter() - run_started
assert len(results) == len(EVALUATION_SEEDS) * EPISODES_PER_SEED * len(methods)
assert all(record["steps"] <= MAX_EPISODE_STEPS for record in results)

paired_starts = {}
for record in results:
    paired_starts.setdefault((record["evaluation_seed"], record["start_id"]), set()).add(record["start"])
assert all(len(starts) == 1 for starts in paired_starts.values())
print(f"Completed {len(results)} episodes in {elapsed_seconds:.3f} seconds.")
print(f"Unique paired starts: {len(paired_starts)}")


In [ ]:
def records_for(method, evaluation_seed=None):
    return [
        row for row in results
        if row["method"] == method
        and (evaluation_seed is None or row["evaluation_seed"] == evaluation_seed)
    ]

seed_summaries = []
for method in methods:
    for evaluation_seed in EVALUATION_SEEDS:
        group = records_for(method, evaluation_seed)
        seed_summaries.append({
            "method": method,
            "seed": evaluation_seed,
            "success_rate": statistics.mean(row["success"] for row in group),
            "mean_steps": statistics.mean(row["steps"] for row in group),
            "mean_model_calls": statistics.mean(row["model_calls"] for row in group),
        })

print("Per-seed results")
print("method          seed   success   mean steps   mean model calls")
for row in seed_summaries:
    print(f"{row['method']:<15} {row['seed']:>4}   {row['success_rate']:>6.1%}    {row['mean_steps']:>8.2f}       {row['mean_model_calls']:>10.1f}")

print("\nAggregate results (mean ± sample SD across five evaluation seeds)")
print("method          success rate       mean steps   model calls/episode   selected-action error   off-support actions")
for method in methods:
    all_rows = records_for(method)
    seed_rates = [row["success_rate"] for row in seed_summaries if row["method"] == method]
    errors = [row["mean_selected_action_error"] for row in all_rows if row["mean_selected_action_error"] is not None]
    support = [row["off_support_fraction"] for row in all_rows if row["off_support_fraction"] is not None]
    error_text = f"{statistics.mean(errors):.3f}" if errors else "n/a"
    support_text = f"{statistics.mean(support):.1%}" if support else "n/a"
    print(
        f"{method:<15} {statistics.mean(seed_rates):>6.1%} ± {statistics.stdev(seed_rates):.1%}"
        f"      {statistics.mean(row['steps'] for row in all_rows):>6.2f}"
        f"            {statistics.mean(row['model_calls'] for row in all_rows):>8.1f}"
        f"                  {error_text:>6}                {support_text:>6}"
    )

by_case = {
    (row["evaluation_seed"], row["start_id"], row["method"]): row
    for row in results
}
noise_wins = sum(
    by_case[(seed, case_id, "policy_noise")]["success"]
    and not by_case[(seed, case_id, "raw_action")]["success"]
    for seed, case_id in paired_starts
)
action_wins = sum(
    by_case[(seed, case_id, "raw_action")]["success"]
    and not by_case[(seed, case_id, "policy_noise")]["success"]
    for seed, case_id in paired_starts
)
print(f"\nPaired starts: policy-noise-only successes={noise_wins}; raw-action-only successes={action_wins}; n={len(paired_starts)}")
print(f"Planner candidates/decision: {PLANNER_CANDIDATES}; horizon: {PLANNING_HORIZON}; episode limit: {MAX_EPISODE_STEPS}")
print(f"Wall time for this CPU run: {elapsed_seconds:.3f} seconds")


## Read the result carefully

The result measures this constructed toy system only. The model's off-support error was written into the example, the policy was hand-authored, and the reference controller reads the true scalar state. If policy-noise search performs better here, that shows the effect of keeping candidates on this policy's support under this chosen model error. It does not establish a general planning advantage or reproduce the LeWAM paper.

The model-call count is the number of forward-model predictions used to score candidate plans. It is not GPU time or a hardware-matched compute measure. The direct controller needs no forward-model calls because it is given the toy state and rule.

### Try a controlled change

Set `MODEL_BIAS = 0.0` and rerun the notebook. The off-support model error should disappear. Compare the new outcomes with the reported run; do not treat this one change as a general ablation study.

### Limits

- The policy, state, environment, and model error are hand-authored; no JEPA or diffusion model is trained.
- The noise input is one scalar and the policy map is a sigmoid, not the paper's flow-matching action-chunk sampler.
- The planner uses one best-of-N random-shooting pass. It does not implement DS-CEM, DS-MPPI, DS-CMA-ES, or typical-set projection.
- The task exposes state directly and has one-dimensional dynamics. It does not test images, robots, distractors, or the paper's benchmark.
- Evaluation-seed variation is not training-run variation.

See the [local run report](../docs/reproductions/policy-noise-planning.md), [shared evaluation protocol](../docs/EVALUATION_PROTOCOL.md), and [reproduction matrix](../docs/REPRODUCTION_MATRIX.md).
